<a href="https://colab.research.google.com/github/shay-bagaria/resistancemap-za/blob/main/ResistanceMap_ZA_02_Data_Acquisition_and_Ingestion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# CELL 1: Pipeline Authentication & Dependency Verification
# ==============================================================================
import os
import sys
from Bio import Entrez, SeqIO
import pandas as pd

print("Initialising Data Ingestion Engine...")

# Lock in your administrative tracking parameters for database compliance
Entrez.email = "s.bagaria@kearsney.co.za"
Entrez.tool = "ResistanceMapZA_v1"

# Create local data directory to store sequences securely
os.makedirs("data/raw_sequences", exist_ok=True)
print("📁 Directory 'data/raw_sequences' verified.")

In [ ]:
# ==============================================================================
# CELL 2: Genomic Database Query & ID Retrieval
# ==============================================================================

# Formulate the programmatic search query targeting South African HIV-1 pol genes
search_query = '"HIV-1"[Organism] AND "South Africa"[Geo_Location] AND pol[Gene] AND 2015:2026[Publication Date]'

print(f"Executing search query: {search_query}\n")

try:
    # Query the engine to get total matching database records
    search_handle = Entrez.esearch(db="nucleotide", term=search_query, retmax="5000")
    search_results = Entrez.read(search_handle)
    search_handle.close()

    global_id_list = search_results["IdList"]
    total_found = search_results["Count"]

    print(f"✅ Query complete.")
    print(f"   Total matching South African sequences in database: {total_found}")
    print(f"   Successfully indexed {len(global_id_list)} record IDs for target extraction.")

except Exception as e:
    print(f"❌ Error during remote database indexing: {e}", file=sys.stderr)

In [ ]:
# ==============================================================================
# CELL 3: Batch Data Fetching & KZN Geographic Filtering
# ==============================================================================

output_fasta_path = "data/raw_sequences/kzn_hiv_pol.fasta"
kzn_matches = []
batch_size = 50  # Balanced batch size to respect NCBI traffic limitations

print(f"Commencing batch download and spatial filtering (Processing {len(global_id_list)} records)...")

# Process records in structured chunks
for i in range(0, len(global_id_list), batch_size):
    sub_list = global_id_list[i:i+batch_size]
    print(f" 📦 Fetching batch indices {i} to {i+len(sub_list)}...")

    try:
        # Pull records in full GenBank text format to extract metadata properties
        fetch_handle = Entrez.efetch(db="nucleotide", id=sub_list, rettype="gb", retmode="text")
        records = list(SeqIO.parse(fetch_handle, "genbank"))
        fetch_handle.close()

        for record in records:
            # Scan all annotations and source text features for regional keywords
            metadata_string = str(record.annotations) + str(record.features).lower()

            if any(kwn in metadata_string for kwn in ["kwazulu", "kzn", "durban", "natal"]):
                kzn_matches.append(record)

    except Exception as e:
        print(f"   ⚠️ Exception encountered in batch processing slot: {e}")
        continue

# Save successfully filtered regional sequences to a clean local file
if kzn_matches:
    SeqIO.write(kzn_matches, output_fasta_path, "fasta")
    print(f"\n✅ Data extraction cycle finalized.")
    print(f"   Total regional sequences successfully isolated: {len(kzn_matches)}")
    print(f"   Clean dataset exported to: {output_fasta_path}")
else:
    print("\n⚠️ Extraction complete, but no explicit KZN metadata markers were found in this sample batch.")